# Repository edition: 04_gate_experiments

Run from this repository in your existing GPU environment. Source is in `../suites/04_gate_experiments/`. The launcher creates a versioned working copy. No embedded base64 code or installation is needed. Review configuration before launching; optional stages are not all enabled by default. See `../docs/EXPERIMENTS.md` for paper mapping.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
SUPPLEMENTARY = next((p for p in _candidates if (p / "tools" / "notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    SUPPLEMENTARY = next((p / "supplementary" for p in _candidates if (p / "supplementary/tools/notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    raise FileNotFoundError("Open this notebook from inside the extracted supplementary repository.")
sys.path.insert(0, str(SUPPLEMENTARY / "tools"))
from notebook_sources import source_bundle as _source_bundle


# Three Ways — full gate experiments

**Uses your existing environment and saved Hugging Face credentials. No installation, environment setup, kernel registration or token cells.** All experiment source is embedded in this notebook.

Upload this notebook to Lambda and run cells in order. It reads the **original `causal_v1` results**; the compact `followup_v1_results.zip` alone does not contain the necessary prompts and cohort records.

The default runs all six experiment families on the full eligible cohort: three-paraphrase variance; exact local three-factor profiles; gate-distance and donor-preactivation interventions; live weight edits with controls and specificity; attention-pattern interventions; and OLMo checkpoint trajectories. The gradient-subspace extension needs genuine training gradients and is explicitly skipped without them.

**This is a substantial campaign, not a smoke test.** Full frozen caches and per-neuron arrays can use tens of GB or more, and checkpoint downloads add storage. Live interventions plus specificity checks may require many GPU-hours or days. The notebook prints free disk and saves resumable results.

Interpretation is deliberately limited: SiLU has no hard opening threshold; exact local frozen contributions do not sum into disjoint origins across layers; pattern patches do not isolate Q/K geometry; and equal weight norms do not imply equal functional costs. The rank-one value baseline is not a full ROME implementation.

In [ ]:
# 1. Settings — full cohort, all six experiment families.
RUN_NAME = "gates_v1"
INPUT_RESULTS = None  # auto-find your original causal_v1 result directory/ZIP
PYTHON_OVERRIDE = None  # normally unnecessary; existing environment is detected
STAGE = "all"  # "all", "core" (1–5), "training" (6), or "report"
RUN_IN_BACKGROUND = False
OVERRIDES = {}
# Change settings only with a new RUN_NAME. No example limit is applied by default.
# Example if full-program cache disk use is too high:
# OVERRIDES = {"save_frozen_cache": False}
# This still saves the requested gate vectors, attention rows and per-neuron factors.


## 2. Prepare the experiment source
The source ZIP is constructed from the readable files in this repository. Its checksum is verified before extraction into a versioned directory. It does not install anything or modify your previous experiment code.

In [ ]:
import ast, base64, hashlib, io, json, os, pathlib, shutil, subprocess, sys, zipfile
WORK = pathlib.Path.cwd().resolve()
BUNDLE = _source_bundle('04_gate_experiments')
EXPECTED_SHA256 = hashlib.sha256(base64.b64decode(_source_bundle('04_gate_experiments'))).hexdigest()
raw = base64.b64decode(BUNDLE)
assert hashlib.sha256(raw).hexdigest() == EXPECTED_SHA256
CODE = WORK / ("threeways_gate_code_" + EXPECTED_SHA256[:12])
CODE.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for name in z.namelist():
        target = (CODE / name).resolve()
        if CODE.resolve() not in target.parents:
            raise RuntimeError("Unsafe archive path")
        if target.exists() and target.read_bytes() != z.read(name):
            raise RuntimeError("Extracted code has local changes. Preserve them and choose a new source directory.")
    z.extractall(CODE)
print("Code:", CODE)
print("Protocol guide:", CODE / "README.txt")
print("Free disk GiB:", round(shutil.disk_usage(WORK).free / 2**30, 1))


## 3. Select and check the existing Python environment
This cell only checks your installed software. It does not create an environment, install packages or request a token. It prefers the environment from the previous follow-up notebook.

The numerical implementation targets Transformers 4.48.3. The tests below verify the adapters in the selected environment before the scientific run.

In [ ]:
if PYTHON_OVERRIDE:
    PYTHON = pathlib.Path(PYTHON_OVERRIDE).expanduser()
else:
    candidates = [WORK / "threeways_followup_env/bin/python",
                  pathlib.Path.home() / "threeways_followup_env/bin/python",
                  WORK / "threeways_causal_env/bin/python",
                  pathlib.Path.home() / "threeways_causal_env/bin/python",
                  pathlib.Path(sys.executable)]
    PYTHON = next(p for p in candidates if p.exists())
print("Using existing Python:", PYTHON)
check = """import sys,torch,transformers,numpy,pandas,scipy,matplotlib,huggingface_hub,tokenizers,pytest
assert transformers.__version__ == '4.48.3', 'Select the previous follow-up environment (Transformers 4.48.3)'
if sys.argv[1] != 'report':
    assert torch.cuda.is_available(), 'CUDA unavailable in the selected environment'
print('Torch:',torch.__version__,'Transformers:',transformers.__version__)
if torch.cuda.is_available():
    print('GPU:',torch.cuda.get_device_name(0))
    print('GPU memory GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,1))
"""
subprocess.run([str(PYTHON), "-c", check, STAGE], check=True)


## 4. Locate your original cohort and save configuration
Leave the defaults for the complete study. `STAGE="all"` runs experiments 1–5 and then the OLMo trajectories. You can schedule `core` and `training` separately with the same configuration and run directory.

Three-factor neuron selection requires independent successful calibration subjects or a verified same-fact donor. Coverage is recorded when neither exists; arbitrary neurons are not substituted.

In [ ]:
if INPUT_RESULTS is None:
    candidates = [WORK / "threeways_causal_results/causal_v1", WORK / "causal_v1_results.zip",
                  pathlib.Path.home() / "threeways_causal_results/causal_v1",
                  pathlib.Path.home() / "causal_v1_results.zip"]
    INPUT = next((p for p in candidates if p.exists()), None)
    if INPUT is None:
        raise FileNotFoundError("Put causal_v1_results.zip next to this notebook, or set INPUT_RESULTS to the original results directory. The followup ZIP alone is insufficient.")
else:
    INPUT = pathlib.Path(INPUT_RESULTS).expanduser().resolve()
    if not INPUT.exists():
        raise FileNotFoundError(INPUT)
CONFIG = json.loads(subprocess.check_output([str(PYTHON), "-c",
    "import json; from gate_study.common import DEFAULT; print(json.dumps(DEFAULT))"], cwd=CODE, text=True))
CONFIG.update(OVERRIDES)
RESULTS = WORK / "threeways_gate_results" / RUN_NAME
CONFIG_PATH = WORK / (RUN_NAME + "_gate_config.json")
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2))
print("Input:", INPUT)
print("Results:", RESULTS)
print("Stage:", STAGE)
print(json.dumps(CONFIG, indent=2))


## 5. Verify the implementation
Small random models test finite neuron deletion, live parameter and activation edits, attention probability constraints, exact rollback, variance identities, cache capture, reports and resume. These tests do not replace the full cohort run.

In [ ]:
subprocess.run([str(PYTHON), "-m", "pytest", "tests", "-q"], cwd=CODE, check=True)


## 6. Run the experiments
Completed prompts, weight-edit arms and checkpoint cases are saved atomically. Rerun this cell with unchanged configuration to resume. Incomplete activation/routing cases restart that case; completed weight arms are retained.

The worker runs independently and writes to disk. With `RUN_IN_BACKGROUND=False`, the notebook monitors it until completion. Interrupting monitoring leaves the worker running; use the next cell before launching another copy. A filesystem lock rejects concurrent writers.

The weight-edit phase may spend substantial time on one case because it evaluates multiple parameter families, doses, random controls and other-subject specificity probes. The monitor counts saved edit files even between case-level progress messages.

In [ ]:
import time
RESULTS.mkdir(parents=True, exist_ok=True)
LOG = RESULTS / "execution.log"
PID_FILE = RESULTS / "worker.pid"
STATUS = RESULTS / (STAGE + "_exit_code.txt")
if PID_FILE.exists():
    pid = int(PID_FILE.read_text())
    try:
        os.kill(pid, 0)
    except ProcessLookupError:
        PID_FILE.unlink()
    else:
        raise RuntimeError(f"Worker {pid} may still be active. Check the monitor before launching again.")
STATUS.unlink(missing_ok=True)
COMMAND = [str(PYTHON), "-u", "-m", "gate_study.run", "--input", str(INPUT),
           "--out", str(RESULTS), "--config", str(CONFIG_PATH), "--stage", STAGE]
supervisor = """import os,pathlib,subprocess,sys
pid=pathlib.Path(sys.argv[1]);status=pathlib.Path(sys.argv[2]);pid.write_text(str(os.getpid()))
try:
    result=subprocess.call(sys.argv[3:]);status.write_text(str(result))
finally:
    pid.unlink(missing_ok=True)
sys.exit(result)
"""
offset = LOG.stat().st_size if LOG.exists() else 0
with LOG.open("a") as log:
    worker = subprocess.Popen([str(PYTHON), "-c", supervisor, str(PID_FILE), str(STATUS), *COMMAND],
        cwd=CODE, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print("Worker PID:", worker.pid)
print("Log:", LOG)
if not RUN_IN_BACKGROUND:
    try:
        with LOG.open() as log:
            log.seek(offset)
            while worker.poll() is None:
                chunk = log.read()
                if chunk:
                    print(chunk, end="", flush=True)
                time.sleep(1)
            print(log.read(), end="")
        if worker.returncode:
            raise RuntimeError(f"Worker exited with code {worker.returncode}. Inspect {LOG}. Completed results are retained.")
    except KeyboardInterrupt:
        print("Monitoring stopped; the worker continues. PID:", worker.pid)


## 7. Monitor progress and inspect results
Start with prompt/neuron-set coverage and numerical certificates. The primary intervention outcome is gold versus the strongest current competitor, plus gold top-1. Complete-answer generation is conditional on gold top-1 and is labelled separately.

`olmo_training/reports/gradient_status.json` records the missing-gradient limitation. It does not prevent the checkpoint trajectories from running.

In [ ]:
from collections import deque
from IPython.display import FileLink, display
if LOG.exists():
    with LOG.open() as f:
        print("".join(deque(f, maxlen=25)))
print("PID:", PID_FILE.read_text() if PID_FILE.exists() else "none recorded")
print("Stage exit code:", STATUS.read_text() if STATUS.exists() else "not recorded yet")
print("Prompt records:", len(list(RESULTS.glob("*/features/*/*.json"))))
print("Saved weight-edit files:", len(list(RESULTS.glob("*/weight_edits/*/*.json"))))
print("Saved training cases:", len(list(RESULTS.glob("olmo_training/step_*/*.json"))))
print("Free disk GiB:", round(shutil.disk_usage(WORK).free / 2**30, 1))
for p in sorted(RESULTS.glob("*/reports/*.pdf")):
    display(FileLink(str(p)))


## 8. Optional report regeneration
Run this only after the worker finishes. It uses saved measurements and interventions; no model weights are loaded.

In [ ]:
REBUILD_REPORTS = False
if REBUILD_REPORTS:
    subprocess.run([str(PYTHON), "-m", "gate_study.run", "--input", str(INPUT),
                    "--out", str(RESULTS), "--config", str(CONFIG_PATH), "--stage", "report"],
                   cwd=CODE, check=True)


## 9. Export a compact results ZIP
Includes the exact source, configurations, selection references, intervention records, per-case profiles, summaries and training trajectories. Large frozen caches, feature arrays, native baseline logits and downloaded model weights stay on the machine. Keep them for future reanalysis.

The default requires the selected stage to have completed. For an intentional partial snapshot only, set `ALLOW_PARTIAL=True`. The archive name records which stage was run.

In [ ]:
ALLOW_PARTIAL = False
complete = STATUS.exists() and STATUS.read_text().strip() == "0" and not PID_FILE.exists()
if not complete and not ALLOW_PARTIAL:
    raise RuntimeError("Selected stage is not complete. Check the log or wait; use ALLOW_PARTIAL=True only for a deliberate partial snapshot.")
ARCHIVE = WORK / (RUN_NAME + "_" + STAGE + ("_results.zip" if complete else "_PARTIAL.zip"))
excluded = {"frozen_cache", "features", "native_baselines", "inputs"}
with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULTS.rglob("*")):
        relative = p.relative_to(RESULTS)
        if p.is_file() and not (set(relative.parts) & excluded) and p.name != ".lock":
            z.write(p, relative)
    z.write(CONFIG_PATH, "configuration.json")
    for p in sorted(CODE.rglob("*")):
        if p.is_file() and not ({"__pycache__", ".pytest_cache"} & set(p.parts)):
            z.write(p, "source/" + str(p.relative_to(CODE)))
print("Saved:", ARCHIVE)
display(FileLink(str(ARCHIVE)))


## Interpretation and references
The hypotheses are tested, not assumed. A successful gate edit does not establish exclusive fact storage in gate rows, and a negative value-edit result at one norm does not rule out value mechanisms. Report specificity, available reference subjects, and fixed dose grids alongside efficacy.

[Knowledge Neurons (Dai et al., 2022)](https://aclanthology.org/2022.acl-long.581/) already uses output-dependent attribution. The distinction here is explicit local factor accounting conditional on a frozen computation, with separate live interventions. The value-side control is a rank-one baseline inspired by editing work, not [ROME](https://rome.baulab.info/) itself. Published OLMo revisions are documented in the [official model card](https://huggingface.co/allenai/OLMo-2-0425-1B).

See the bundled `README.txt` for equations, reference selection, coordinate alignment, endpoints, all output files, and the optional actual-gradient manifest schema.